In [ ]:
import requests

: 

In [ ]:
request = requests.get("https://www.basketball-reference.com/teams/MIA/2026.html")
print(request.status_code)

In [ ]:
import pandas as pd

def get_team_players(url: str):
    # Read all tables on the page
    tables = pd.read_html(url)

    roster_df = None

    # Find the roster table (the one with a 'Player' column and reasonable size)
    for df in tables:
        if 'Player' in df.columns and len(df) <= 30:
            roster_df = df
            break

    if roster_df is None:
        raise ValueError("Could not find roster table with a 'Player' column.")

    # Get unique player names as a list
    players = roster_df['Player'].dropna().unique().tolist()
    return roster_df

if __name__ == "__main__":
    temp_team = "MIA"
    url = f"https://www.basketball-reference.com/teams/{temp_team}/2026.html"
    players = get_team_players(url)

    for name in players:
        print(name)


In [ ]:
roster_df = get_team_players(url)
roster_df.head()

In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup, Comment
from selenium.webdriver.chrome.service import Service

url = "https://www.basketball-reference.com/teams/MIA/2026.html"

# Start Chrome
driver = webdriver.Chrome()
driver.get(url)

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, "html.parser")

# Basketball-reference hides tables in comments sometimes


In [ ]:
tables = soup.find("table", {"id": "roster"})
tables

In [ ]:
rows =tables.find("tbody").find_all("tr")
rows


In [ ]:
from bs4 import BeautifulSoup

player = {}
row = rows[0]
# row = <your tr bs4 element>

# jersey number
player["number"] = row.find("th", {"data-stat": "number"}).get_text(strip=True)

# POS
player["pos"] = row.find("td", {"data-stat": "pos"}).get_text(strip=True)

# player cell
player_cell = row.find("td", {"data-stat": "player"})

player["name"] = player_cell.get_text(strip=True)

# full href
player["href"] = "https://www.basketball-reference.com" + player_cell.find("a")["href"]

# birth date
player["birth_date"] = row.find("td", {"data-stat": "birth_date"}).get_text(strip=True)

print(player)


In [ ]:
roster_table

In [ ]:
players

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.basketball-reference.com/teams/"
html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

teams = []

team_row = soup.find("table", {"id": "teams_active"})

#rows = team_row.find("tbody").find_all("tr")

team_row

In [ ]:
soup.find("table", {"id": "teams_active"})

In [ ]:
soup.find_all("table")

In [ ]:
html

# progran begins


In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup, Comment
from selenium.webdriver.chrome.service import Service
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()

# Retry logic (handles random blocks / network hiccups)
retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

import json
import os

PROGRESS_FILE = "data/progress.json"

def load_progress():
    if not os.path.exists(PROGRESS_FILE):
        return set()
    with open(PROGRESS_FILE, "r") as f:
        data = json.load(f)
    # stored as list of [year, team], convert back to set of tuples
    return { (int(y), t) for y, t in data }

def save_progress(done_set):
    os.makedirs("data", exist_ok=True)
    # convert set of tuples back to list of lists for json
    data = [[y, t] for (y, t) in sorted(done_set)]
    with open(PROGRESS_FILE, "w") as f:
        json.dump(data, f)


def get_page_source(url: str, retries: int = 3, delay: int = 5) -> str:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    import time
    
    """
    Load a page with headless Chrome, with retries on Selenium/driver errors.
    Returns HTML string, or "" if all attempts fail.
    """
    for attempt in range(1, retries + 1):
        options = Options()
        options.add_argument("--headless=new")   # run Chrome in headless mode
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--window-size=1920,1080")

        driver = webdriver.Chrome(options=options)
        try:
            driver.set_page_load_timeout(60)  # seconds
            print(f"[TRY {attempt}/{retries}] Accessing {url}")
            driver.get(url)
            html = driver.page_source
            return html
        except Exception as e:
            print(f"Error accessing {url} on attempt {attempt}: {e}")
            if attempt == retries:
                print(f"[FAIL] Giving up on {url}")
                return ""
            time.sleep(delay)
        finally:
            driver.quit()

def get_teams():
    import pandas as pd
    import os

    os.makedirs("data", exist_ok=True)
    url = f"https://www.basketball-reference.com/teams/"
    html = get_page_source(url)
    if not html:
        raise ValueError("Could not retrieve the page source.")
    
    soup = BeautifulSoup(html, "html.parser")

    teams_table = soup.find("table", {"id": "teams_active"})
    if not teams_table:
        raise ValueError("Could not find teams table.")

    rows = teams_table.find("tbody").find_all("tr")
    teams = []

    for row in rows:
        # some rows can be header separators, be safe:
        name_cell = row.find("th", {"data-stat": "franch_name"})
        if not name_cell:
            continue

        link = name_cell.find("a")
        if not link:
            continue

       
        team = {
            "name": link.get_text(strip=True),
            "href": link["href"].split("/")[2]
,
        }
        teams.append(team)

    pd.DataFrame(teams).to_csv("data/teams.csv", index=False)
    return teams

def parse_roster_table(html: str, team, year,) -> list:
    soup = BeautifulSoup(html, "html.parser")

    # Find the roster table
    roster_table = soup.find("table", {"id": "roster"})

    if not roster_table:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for c in comments:
            if 'id="roster"' in c:
                roster_table = BeautifulSoup(c, "html.parser").find("table", id="roster")
                break

    # if not roster_table:
    #     raise ValueError(f"Could not find roster table for {team}.")
    
    if not roster_table:
            print(f"[WARN] No roster table for team {team} : {year}. Skipping.")
            return []

    # Extract rows from the roster table
    rows = roster_table.find("tbody").find_all("tr")

    players = []
    for row in rows:
        player = {}

        # Player name and href
        player_cell = row.find("td", {"data-stat": "player"})
        player["name"] = player_cell.get_text(strip=True)

        # POS

        player["pos"] = row.find("td", {"data-stat": "pos"}).get_text(strip=True)

        # Team
        player["team"] = team

        # Jersey number
        player["number"] = row.find("th", {"data-stat": "number"}).get_text(strip=True)

       
        player["href"] = "https://www.basketball-reference.com" + player_cell.find("a")["href"]

        # Birth date
        player["birth_date"] = row.find("td", {"data-stat": "birth_date"}).get_text(strip=True)

        players.append(player)

    return players


def initialize_players(start_year: int = 2024, end_year: int = 2025):
    import pandas as pd
    import time
    import os

    os.makedirs("data", exist_ok=True)

    try:
        teams_df = pd.read_csv("data/teams.csv")
    except FileNotFoundError:
        teams_df = get_teams()
        teams_df = pd.DataFrame(teams_df)  # because get_teams currently returns list

    teams = teams_df['href'].tolist()

    # ✅ load which (year, team) combos are already done
    done = load_progress()
    print(f"Loaded {len(done)} completed (year, team) combos from progress file.")

    # ✅ if players.csv exists, load & keep appending; else start fresh
    players_path = "data/players.csv"
    if os.path.exists(players_path):
        players_df = pd.read_csv(players_path)
        players = players_df.to_dict(orient="records")
        print(f"Loaded {len(players)} existing players from {players_path}.")
    else:
        players = []

    for year in range(start_year, end_year):
        for i, team in enumerate(teams):
            if (year, team) in done:
                print(f"[SKIP] Already processed {team} {year}")
                continue

            print(f"Processing team {i+1}/{len(teams)}: {team} for year {year}")
            url = f"https://www.basketball-reference.com/teams/{team}/{year}.html"
            html = get_page_source(url)

            if not html:
                print(f"[WARN] Empty HTML for {team} {year}. Skipping.")
                # still mark as done so we don't hammer same broken page forever
                done.add((year, team))
                save_progress(done)
                continue

            new_players = parse_roster_table(html, team, year)
            players.extend(new_players)

            # ✅ save progress after each team to be safe
            pd.DataFrame(players).drop_duplicates(subset=['href']).to_csv(players_path, index=False)
            done.add((year, team))
            save_progress(done)

            time.sleep(3)

    print(f"Finished. Total players: {len(players)}")


# test

In [ ]:
end_year = 2026
start_year = end_year - 7
initialize_players(start_year, end_year)

# real

In [ ]:
end_year = 2026
start_year = end_year - 7
initialize_players(start_year, end_year)

In [ ]:
import time
teams = ["NJN", "MIA"]
year = 2026
players = []
for i, team in enumerate(teams):
        print(f"Processing team {i+1}/{len(teams)}: {team}")
        url = f"https://www.basketball-reference.com/teams/{team}/{year}.html"
        html = get_page_source(url)
        # print(html)
        new_players = parse_roster_table(html, team, year)
        players.extend(new_players)
        time.sleep(3)

In [ ]:
players

# game logs

In [ ]:
import pandas as pd
data = pd.read_csv("data/players.csv")
test = data[data['name'] == 'Jimmy Butler'].copy()
test['href'].values[0]
test["url_gl"] = "https://www.basketball-reference.com/players/b/butleji01/gamelog/2026"

test["href"].values[0].split(".html")[0] + "/gamelog/2026"
d2 =data.copy()
d2["url_gl"] = d2["href"].str.split(".html").str[0] + "/gamelog/2026"
d2.iloc[301]["url_gl"]


In [ ]:
from bs4 import BeautifulSoup, Comment
from nba_scraper.browser import get_page_source

def parse_gamelog_page(html: str, player: str, href: str, season: int) -> list[dict]:
    """
    Parse a Basketball-Reference game log page like:
    https://www.basketball-reference.com/players/c/chrisma01/gamelog/2017
    
    Returns a list of dicts, one per game.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Find table normally
    table = soup.find("table", id="player_game_log_reg")

    # Fallback if hidden in comments
    if not table:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for c in comments:
            if 'id="player_game_log_reg"' in c:
                table = BeautifulSoup(c, "html.parser").find("table", id="player_game_log_reg")
                break

    if not table:
        print(f"[WARN] No game log table for {href} {season}")
        return []

    rows = table.find("tbody").find_all("tr")
    games = []

    def get_stat(row, stat_name: str) -> str:
        cell = row.find("td", {"data-stat": stat_name})
        return cell.get_text(strip=True) if cell else ""

    for row in rows:
        # real game rows only
        if row.get("data-row") is None:
            continue

        game = {
            "player": player,  
            "date":        get_stat(row, "date"),
            "team":        get_stat(row, "team_name_abbr"),
            "opp":         get_stat(row, "opp_name_abbr"),
            "home_away":   get_stat(row, "game_location"),  # '' or '@'
            "result":      get_stat(row, "game_result"),
            "gs":          get_stat(row, "gs"),             # games started

            "mp":   get_stat(row, "mp"),

            "fg":   get_stat(row, "fg"),
            "fga":  get_stat(row, "fga"),
            "fg3":  get_stat(row, "fg3"),
            "fg3a": get_stat(row, "fg3a"),
            "ft":   get_stat(row, "ft"),
            "fta":  get_stat(row, "fta"),

            "orb":  get_stat(row, "orb"),
            "drb":  get_stat(row, "drb"),
            "trb":  get_stat(row, "trb"),

            "ast":  get_stat(row, "ast"),
            "stl":  get_stat(row, "stl"),
            "blk":  get_stat(row, "blk"),
            "tov":  get_stat(row, "tov"),
            "pf":   get_stat(row, "pf"),
            "pts":  get_stat(row, "pts"),
            "href": href,          # player identity via profile URL
            "season": season,
        }

        games.append(game)

    return games

import os
import json

GAMELOG_PROGRESS_FILE = "data/progress/progress_gamelogs.json"

def load_gamelog_progress():
    if not os.path.exists(GAMELOG_PROGRESS_FILE):
        return set()
    with open(GAMELOG_PROGRESS_FILE, "r") as f:
        data = json.load(f)
    # stored as list of [season, href]
    return {(int(season), href) for season, href in data}

def save_gamelog_progress(done_set):
    os.makedirs(os.path.dirname(GAMELOG_PROGRESS_FILE), exist_ok=True)

    data = [[season, href] for (season, href) in sorted(done_set)]
    with open(GAMELOG_PROGRESS_FILE, "w") as f:
        json.dump(data, f)

import pandas as pd
import os
import time
from pandas.errors import EmptyDataError

def get_gamelogs(data: pd.DataFrame | None = None, year: int = 2026, debug: bool = False):
    os.makedirs("data", exist_ok=True)

    # Load player list if not provided
    if data is None:
        data = pd.read_csv("data/players.csv")

    # Load gamelog progress
    done = load_gamelog_progress()
    print(f"Loaded {len(done)} completed (season, href) combos from gamelog progress.")

    # Determine output CSV for this season
    out_path = f"data/gamelogs/gamelogs_{year}.csv"

    # If CSV already exists, load existing games so we don't lose them
    games = []
    
    if os.path.exists(out_path):
        try:
            existing_df = pd.read_csv(out_path)
            games = existing_df.to_dict(orient="records")
            print(f"Loaded {len(games)} existing game rows from {out_path}.")
        except EmptyDataError:
            print(f"[WARN] {out_path} exists but is empty. Starting fresh.")
            

    i = 0
    for row in data.itertuples(index=False):
        print(f"{i}/{len(data)}")

        i += 1

        print(f"Processing player: {row.name}")
        href = row.href
        player = row.name

        # Skip if this (season, href) is already done
        if (year, href) in done:
            print(f"[SKIP] Already processed {href} {year}")
            continue

        url = href.replace(".html", f"/gamelog/{year}")
        print(f"Scraping gamelog: {url}")

        html = get_page_source(url)
        if not html:
            print(f"[WARN] Empty HTML for {href} {year}. Skipping.")
            # still mark as done so we don't keep retrying a dead page forever
            done.add((year, href))
            save_gamelog_progress(done)
            continue

        new_games = parse_gamelog_page(html=html, player=row.name, href=href, season=year)
    
        games.extend(new_games)

        # Save incrementally after each player
        pd.DataFrame(games).to_csv(out_path, index=False)
        done.add((year, href))
        save_gamelog_progress(done)

        # optional delay to be polite
        time.sleep(1)

        if debug and new_games:
            print("[DEBUG] Stopping after first player with data.")
            break

    print(f"Finished gamelog scraping for season {year}. Total games: {len(games)}")
    return games

get_gamelogs(year=2026, debug=False)

# test

In [ ]:
data = pd.read_csv("data/players.csv")
test_df = data[data['name'] == 'Jimmy Butler'].copy()
url = test_df['href'].values[0].split(".html")[0] + "/gamelog/2026"
html = get_page_source(url)
games = parse_gamelog_page(html=html, player=test_df['name'].values[0], href=test_df['href'].values[0], season=2026)

In [ ]:
soup = BeautifulSoup(html, "html.parser")
soup.find_all("h1")

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)

pd.DataFrame(games).head()

In [ ]:
test_df = data[data['name'] == 'Jimmy Butler'].copy()
url = test_df['href'].values[0].split(".html")[0] + "/gamelog/2026"
html = get_page_source(url)
soup = BeautifulSoup(html, "html.parser")

soup.find("table", id="player_game_log_reg")

In [ ]:
import pandas as pd
data = pd.read_csv("data/players.csv")
data = data[data['name'].str.contains("Jimmy Butler")].copy()
href = data.iloc[0]['href']
year = 2026
url = href.replace(".html", f"/gamelog/{year}")
print(f"Scraping gamelog: {url}")

html = get_page_source(url)
if not html:
    print(f"[WARN] Empty HTML for {href} {year}. Skipping.")
    # still mark as done so we don't keep retrying a dead page forever
    

new_games = parse_gamelog_page(html, href=href, season=year)
if new_games:
    new_games["player"] = data.iloc[0]['name']

new_games


In [ ]:
data['player_slug'] = data['href'][0].split("/")[5].split(".")[0]

# data 

In [1]:
import pandas as pd


import os


In [1]:
import pandas as pd
import os
for dir,_,files in os.walk("data/gamelogs/"):
    datas = []
    for file in files:
        if file.endswith(".csv"):
            path = os.path.join(dir, file)
            print(f"Processing {path}")
            df = pd.read_csv(path)
            datas.append(df)


Processing data/gamelogs/gamelogs_2021.csv
Processing data/gamelogs/gamelogs_2022.csv
Processing data/gamelogs/gamelogs_2023.csv
Processing data/gamelogs/gamelogs_2024.csv
Processing data/gamelogs/gamelogs_2025.csv
Processing data/gamelogs/gamelogs_2026.csv


In [ ]:
def min_played_minutes(row):
    """Convert 'mm:ss' string to total minutes as float."""
    if pd.isna(row):
        return 0.0
    try:
        minutes, seconds = map(int, row.split(':'))
        total_minutes = minutes + seconds / 60.0
        return total_minutes
    except Exception as e:
        print(f"Error parsing mp '{row}': {e}")
        return 0.0
    


In [3]:
import pandas as pd
import pandas as pd
import os
for dir,_,files in os.walk("data/gamelogs/"):
    datas = []
    for file in files:
        if file.endswith(".csv"):
            path = os.path.join(dir, file)
            print(f"Processing {path}")
            df = pd.read_csv(path)
            datas.append(df)


data_ori =pd.concat(datas, ignore_index=True)
print(f"Total gamelog rows combined: {len(data_ori)}")
drop_cols = ["gs", "home_away", "href", 'result']
data = data_ori.copy()
data.dropna(subset=["mp"], inplace=True)
data['is_home'] =data["home_away"].apply(lambda x: 1 if x == "@" else 0)
data["is_win"] = data["result"].apply(lambda x: 1 if x.startswith("W") else 0)
data["date"] = pd.to_datetime(data["date"])
data["mp_minutes"] = data["mp"].apply(min_played_minutes)
data['usage'] = data['mp_minutes'] / 48.0
data.drop(drop_cols, axis=1, inplace=True)
data.drop_duplicates(inplace=True, subset=['player', 'date', 'season'], keep='last')
data.to_csv("data/all_gamelogs_combined.csv", index=False)
print("the lastest date in the data is:", data['date'].max())
print("the total number of unique players is:", data['player'].nunique())
print("the total number of gamelog rows is:", len(data))
data.head()

Processing data/gamelogs/gamelogs_2021.csv
Processing data/gamelogs/gamelogs_2022.csv
Processing data/gamelogs/gamelogs_2023.csv
Processing data/gamelogs/gamelogs_2024.csv
Processing data/gamelogs/gamelogs_2025.csv
Processing data/gamelogs/gamelogs_2026.csv
Total gamelog rows combined: 83348
the lastest date in the data is: 2026-01-22 00:00:00
the total number of unique players is: 802
the total number of gamelog rows is: 46532


,player,season,date,team,opp,mp,fg,fga,fg3,fg3a,...,ast,stl,blk,tov,pf,pts,is_home,is_win,mp_minutes,usage
0,Nathan Knight,2021,2020-12-23,ATL,CHI,05:04,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,2.0,2.0,0.0,1,1,5.066667,0.105556
1,Nathan Knight,2021,2020-12-26,ATL,MEM,08:35,4.0,5.0,2.0,3.0,...,0.0,0.0,0.0,1.0,1.0,14.0,1,1,8.583333,0.178819
5,Nathan Knight,2021,2021-01-01,ATL,BRK,00:54,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,0.900000,0.018750
8,Nathan Knight,2021,2021-01-06,ATL,CHO,02:45,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0,0,2.750000,0.057292
9,Nathan Knight,2021,2021-01-09,ATL,CHO,02:07,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1,0,2.116667,0.044097


### function getdata and transformdata

In [4]:
import pandas as pd
import pandas as pd
import os

def get_data():
   
    for dir,_,files in os.walk("data/gamelogs/"):
        datas = []
        for file in files:
            if file.endswith(".csv"):
                path = os.path.join(dir, file)
                print(f"Processing {path}")
                df = pd.read_csv(path)
                datas.append(df)

    data =pd.concat(datas, ignore_index=True)
    return data


def transform_date(data):
    print(f"Total gamelog rows combined: {len(data)}")
    drop_cols = ["gs", "home_away", "href", 'result']
    data.dropna(subset=["mp"], inplace=True)
    data['is_home'] =data["home_away"].apply(lambda x: 1 if x == "@" else 0)
    data["is_win"] = data["result"].apply(lambda x: 1 if x.startswith("W") else 0)
    data["date"] = pd.to_datetime(data["date"])
    data["mp_minutes"] = data["mp"].apply(min_played_minutes)
    data['usage'] = data['mp_minutes'] / 48.0
    data.drop(drop_cols, axis=1, inplace=True)
    data.drop_duplicates(inplace=True, subset=['player', 'date', 'season'], keep='last')
    data.to_csv("data/all_gamelogs_combined.csv", index=False)
    print("the lastest date in the data is:", data['date'].max())
    print("the total number of unique players is:", data['player'].nunique())
    print("the total number of gamelog rows is:", len(data))
    return data

def get_transform():
    return transform_date(get_data())
    
data = get_data()
data = transform_date(data)

data = get_transform()


Processing data/gamelogs/gamelogs_2021.csv
Processing data/gamelogs/gamelogs_2022.csv
Processing data/gamelogs/gamelogs_2023.csv
Processing data/gamelogs/gamelogs_2024.csv
Processing data/gamelogs/gamelogs_2025.csv
Processing data/gamelogs/gamelogs_2026.csv
Total gamelog rows combined: 83348
the lastest date in the data is: 2026-01-22 00:00:00
the total number of unique players is: 802
the total number of gamelog rows is: 46532
Processing data/gamelogs/gamelogs_2021.csv
Processing data/gamelogs/gamelogs_2022.csv
Processing data/gamelogs/gamelogs_2023.csv
Processing data/gamelogs/gamelogs_2024.csv
Processing data/gamelogs/gamelogs_2025.csv
Processing data/gamelogs/gamelogs_2026.csv
Total gamelog rows combined: 83348
the lastest date in the data is: 2026-01-22 00:00:00
the total number of unique players is: 802
the total number of gamelog rows is: 46532


In [5]:
data = pd.read_csv("data/all_gamelogs_combined.csv")

In [5]:
df = data[data["season"] == 2026].sort_values(by=['date'], ascending=False, inplace=False).copy()

In [6]:
df_players = df.groupby('player')['usage'].mean().sort_values(ascending=False).head(110).reset_index()
players = df_players["player"].to_list()
df2 = df[df['player'].isin(players)].copy()
df2

,player,season,date,team,opp,mp,fg,fga,fg3,fg3a,...,ast,stl,blk,tov,pf,pts,is_home,is_win,mp_minutes,usage
45602,Donovan Mitchell,2026,2026-01-21,CLE,CHO,37:29,8.0,20.0,2.0,8.0,...,6.0,1.0,0.0,8.0,1.0,24.0,1,1,37.483333,0.780903
46015,Jalen Brunson,2026,2026-01-21,NYK,BRK,30:33,8.0,17.0,1.0,8.0,...,5.0,0.0,0.0,2.0,2.0,20.0,0,1,30.550000,0.636458
46013,Mikal Bridges,2026,2026-01-21,NYK,BRK,24:33,5.0,9.0,1.0,3.0,...,4.0,1.0,1.0,0.0,2.0,11.0,0,1,24.550000,0.511458
45606,Evan Mobley,2026,2026-01-21,CLE,CHO,35:31,6.0,13.0,0.0,4.0,...,0.0,1.0,3.0,2.0,4.0,14.0,1,1,35.516667,0.739931
45743,Pascal Siakam,2026,2026-01-21,IND,BOS,32:43,12.0,21.0,4.0,7.0,...,4.0,0.0,0.0,2.0,3.0,32.0,1,0,32.716667,0.681597
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39536,Luka Dončić,2026,2025-10-21,LAL,GSW,40:59,17.0,27.0,2.0,10.0,...,9.0,2.0,1.0,3.0,1.0,43.0,0,0,40.983333,0.853819
39513,Austin Reaves,2026,2025-10-21,LAL,GSW,36:20,9.0,16.0,1.0,5.0,...,9.0,2.0,0.0,5.0,5.0,26.0,0,0,36.333333,0.756944
38199,Amen Thompson,2026,2025-10-21,HOU,OKC,38:36,8.0,17.0,0.0,7.0,...,5.0,1.0,1.0,4.0,3.0,18.0,1,0,38.600000,0.804167
41740,Shai Gilgeous-Alexander,2026,2025-10-21,OKC,HOU,47:13,12.0,26.0,1.0,9.0,...,5.0,2.0,2.0,3.0,2.0,35.0,0,1,47.216667,0.983681


In [7]:
df2 = df.groupby('player').agg({'usage': 'max'}).sort_values(by=['usage'], ascending=False, inplace=False).head(110).reset_index()
players = df2["player"].to_list()
df3 = data[data['player'].isin(players)].copy()

In [8]:
df3.describe()

,season,fg,fga,fg3,fg3a,ft,fta,orb,drb,trb,ast,stl,blk,tov,pf,pts,is_home,is_win,mp_minutes,usage
count,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000,12531.000000
mean,2024.812545,5.545128,11.690767,1.663874,4.567632,2.549757,3.188971,1.187934,4.099433,5.287367,3.398851,0.974463,0.570585,1.762589,2.149868,15.303886,0.498364,0.527731,28.885281,0.601777
std,1.339339,3.498184,6.215604,1.639064,3.176975,2.912062,3.443843,1.440837,2.961149,3.721899,2.870197,1.078182,0.935865,1.611776,1.454512,9.581385,0.500017,0.499250,8.765692,0.182619
min,2021.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025.000000,3.000000,7.000000,0.000000,2.000000,0.000000,0.000000,0.000000,2.000000,3.000000,1.000000,0.000000,0.000000,1.000000,1.000000,8.000000,0.000000,0.000000,23.900000,0.497917
50%,2025.000000,5.000000,11.000000,1.000000,4.000000,2.000000,2.000000,1.000000,4.000000,5.000000,3.000000,1.000000,0.000000,1.000000,2.000000,14.000000,0.000000,1.000000,30.633333,0.638194
75%,2026.000000,8.000000,16.000000,3.000000,6.000000,4.000000,5.000000,2.000000,6.000000,7.000000,5.000000,2.000000,1.000000,3.000000,3.000000,21.000000,1.000000,1.000000,35.316667,0.735764
max,2026.000000,22.000000,45.000000,12.000000,20.000000,23.000000,26.000000,13.000000,23.000000,28.000000,22.000000,8.000000,10.000000,10.000000,6.000000,61.000000,1.000000,1.000000,56.333333,1.173611


# defense

In [9]:
import pandas as pd
import numpy as np

path = "./data/gamelogs/gamelogs_2026.csv"
df = pd.read_csv(path)

# Ensure numeric
for col in ["fg3", "fg3a"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Aggregate team offensive 3pt stats per game
game_team = (
    df.groupby(["season", "date", "team", "opp"], as_index=False)
      .agg(team_fg3=("fg3", "sum"), team_fg3a=("fg3a", "sum"))
)

# Attribute those made/attempted 3s to the opponent's defense
def_games = game_team.rename(columns={
    "team": "offense_team",
    "opp": "defense_team",
    "team_fg3": "opp_fg3_made",
    "team_fg3a": "opp_fg3_att"
})

# Season defense totals per team
def_season = (
    def_games.groupby(["season", "defense_team"], as_index=False)
    .agg(
        games=("opp_fg3_made", "size"),
        opp_fg3_made_total=("opp_fg3_made", "sum"),
        opp_fg3_att_total=("opp_fg3_att", "sum")
    )
)

def_season["opp_3pm_per_game"] = def_season["opp_fg3_made_total"] / def_season["games"]
def_season["opp_3pa_per_game"] = def_season["opp_fg3_att_total"] / def_season["games"]
def_season["opp_3p_pct_allowed"] = def_season["opp_fg3_made_total"] / def_season["opp_fg3_att_total"]

# Rankings (lower is better)
def_season["rank_by_3pm_per_game"] = (
    def_season.groupby("season")["opp_3pm_per_game"]
    .rank(method="min", ascending=True)
    .astype(int)
)
def_season["rank_by_3p_pct_allowed"] = (
    def_season.groupby("season")["opp_3p_pct_allowed"]
    .rank(method="min", ascending=True)
    .astype(int)
)

def_season_sorted = def_season.sort_values(
    ["season", "rank_by_3pm_per_game", "rank_by_3p_pct_allowed"]
).reset_index(drop=True)

out_path = "./data/defense_ranking_vs_three_2026.csv"
def_season_sorted.to_csv(out_path, index=False)
print("Saved:", out_path)


Saved: ./data/defense_ranking_vs_three_2026.csv


In [10]:
def_season_sorted

,season,defense_team,games,opp_fg3_made_total,opp_fg3_att_total,opp_3pm_per_game,opp_3pa_per_game,opp_3p_pct_allowed,rank_by_3pm_per_game,rank_by_3p_pct_allowed
0,2026,IND,40,445.0,1311.0,11.125000,32.775000,0.339436,1,2
1,2026,ORL,37,426.0,1195.0,11.513514,32.297297,0.356485,2,13
2,2026,DAL,39,462.0,1391.0,11.846154,35.666667,0.332135,3,1
3,2026,PHI,38,469.0,1346.0,12.342105,35.421053,0.348440,4,4
4,2026,GSW,40,497.0,1412.0,12.425000,35.300000,0.351983,5,9
5,2026,PHO,40,499.0,1425.0,12.475000,35.625000,0.350175,6,6
6,2026,TOR,39,489.0,1407.0,12.538462,36.076923,0.347548,7,3
7,2026,DET,39,489.0,1402.0,12.538462,35.948718,0.348787,7,5
8,2026,MIN,39,491.0,1361.0,12.589744,34.897436,0.360764,9,17
9,2026,SAC,45,568.0,1617.0,12.622222,35.933333,0.351268,10,7


In [29]:
import pandas as pd

# Load your data
df = pd.DataFrame()
for i in range(2025, 2026):
    df = pd.concat([df, pd.read_csv(f"./data/gamelogs/gamelogs_{i}.csv")])

# Make sure 3pt columns are numeric
df["fg3"] = pd.to_numeric(df["fg3"], errors="coerce")
df["fg3a"] = pd.to_numeric(df["fg3a"], errors="coerce")

# Group by player vs opponent team
player_vs_team = (
    df.groupby(["player", "opp"], as_index=False)
      .agg(
          games=("date", "nunique"),
          fg3_made=("fg3", "sum"),
          fg3_att=("fg3a", "sum")
      )
)

# Add shooting metrics
player_vs_team["3pm_per_game"] = player_vs_team["fg3_made"] / player_vs_team["games"]
player_vs_team["3p_pct"] = player_vs_team["fg3_made"] / player_vs_team["fg3_att"]

# Optional: sort for easier viewing
player_vs_team = player_vs_team.sort_values(
    ["player", "3pm_per_game"], 
    ascending=[True, False]
)

# Save to CSV
out_path = "./data/player_vs_team_3pt_splits.csv"
player_vs_team.to_csv(out_path, index=False)

out_path


'./data/player_vs_team_3pt_splits.csv'

In [30]:
player_vs_team

,player,opp,games,fg3_made,fg3_att,3pm_per_game,3p_pct
6,A.J. Green,DAL,2,7.0,12.0,3.500000,0.583333
7,A.J. Green,DEN,2,7.0,12.0,3.500000,0.583333
23,A.J. Green,POR,2,7.0,13.0,3.500000,0.538462
16,A.J. Green,MIN,2,6.0,13.0,3.000000,0.461538
28,A.J. Green,WAS,3,8.0,21.0,2.666667,0.380952
...,...,...,...,...,...,...,...
13099,Zyon Pullin,POR,1,0.0,0.0,0.000000,NaN
13100,Zyon Pullin,SAC,1,0.0,0.0,0.000000,NaN
13101,Zyon Pullin,SAS,4,0.0,0.0,0.000000,NaN
13102,Zyon Pullin,TOR,1,0.0,0.0,0.000000,NaN


In [ ]:
data[data['team'] == "MEM".upper()]["player"].unique()


array(['Desmond Bane', 'Sean McDermott', 'Jontay Porter',
       'Killian Tillie', 'Xavier Tillman Sr.', 'Santi Aldama',
       'Shaq Buchanan', 'Yves Pons', 'Xavier Sneed', 'Jon Teske',
       'Ziaire Williams', 'Kennedy Chandler', 'Jacob Gilyard',
       'Jake LaRavia', 'Kenneth Lofton Jr.', 'David Roddy',
       'Vince Williams Jr.', 'Tosan Evbuomwan', 'Timmy Allen',
       'Matthew Hurt', 'GG Jackson II', 'Trey Jemison',
       'Mãozinha Pereira', 'Zach Edey', 'Yuki Kawamura', 'Zyon Pullin',
       'Cam Spencer', 'Jaylen Wells', 'Marvin Bagley III',
       'Colin Castleton', 'Brandon Clarke', 'Jay Huff',
       'Jaren Jackson Jr.', 'Luke Kennard', 'John Konchar', 'Ja Morant',
       'Scotty Pippen Jr.', 'Marcus Smart', 'Lamar Stevens',
       'Cedric Coward', 'Javon Small(TW)', 'Jahmai Mashack(TW)',
       'Kentavious Caldwell-Pope', 'Jock Landale',
       'Olivier-Maxence Prosper(TW)', 'Christian Koloko'], dtype=object)

In [41]:
p = r"Julius Randle"
player_vs_team[player_vs_team["player"] == p]

,player,opp,games,fg3_made,fg3_att,3pm_per_game,3p_pct
6957,Julius Randle,BRK,2,8.0,14.0,4.000000,0.571429
6956,Julius Randle,BOS,2,6.0,11.0,3.000000,0.545455
6955,Julius Randle,ATL,2,5.0,15.0,2.500000,0.333333
6962,Julius Randle,DEN,4,10.0,24.0,2.500000,0.416667
6961,Julius Randle,DAL,3,7.0,10.0,2.333333,0.700000
6977,Julius Randle,PHO,4,9.0,23.0,2.250000,0.391304
6969,Julius Randle,MEM,3,6.0,22.0,2.000000,0.272727
6973,Julius Randle,NYK,2,4.0,10.0,2.000000,0.400000
6979,Julius Randle,SAC,4,8.0,18.0,2.000000,0.444444
6967,Julius Randle,LAC,3,5.0,17.0,1.666667,0.294118


# threes

In [15]:
import pandas as pd
data = pd.read_csv("data/all_gamelogs_combined.csv")
df_three = data[data["season"] == 2026].sort_values(by=['usage'], ascending=False, inplace=False).copy()


df_three = df_three.groupby('player').agg({'usage': 'max'}).sort_values(by=['usage'], ascending=False, inplace=False).head(110).reset_index()
players = df_three["player"].to_list()

df3 = data[data['player'].isin(players)].copy()
df3["fg3_avg"] = df3["fg3"] / df3["fg3a"]
fg3a = df3['fg3a'].mean()
fg3  = df3['fg3'].mean()
avg_fg3_pct = fg3 / fg3a
df3 = df3.groupby('player').agg({'fg3_avg': 'mean', 'fg3a': 'mean'}).sort_values(by=['fg3_avg'], ascending=False, inplace=False).reset_index()
df3[df3['fg3_avg'] > avg_fg3_pct]

df3.head(25)

,player,fg3_avg,fg3a
0,Nikola Jokić,0.463208,4.754902
1,Kevin Durant,0.453615,5.811881
2,Zach LaVine,0.416697,7.093458
3,Shai Gilgeous-Alexander,0.411929,5.403361
4,Daniss Jenkins(TW),0.408929,2.500000
5,Grayson Allen,0.406952,6.516484
6,Ryan Rollins,0.402349,3.369369
7,Royce O'Neale,0.401545,6.285714
8,A.J. Green,0.400124,5.047297
9,Kawhi Leonard,0.399775,6.089552


In [16]:
import pandas as pd

data = pd.read_csv("data/all_gamelogs_combined.csv")

# Keep only 2026 season
df = data[data["season"] == 2026].copy()

# Make sure date is datetime
df["date"] = pd.to_datetime(df["date"])

# Calculate per-game 3PT%
df["fg3_pct"] = df["fg3"] / df["fg3a"]

# Sort so rolling works correctly
df = df.sort_values(["player", "date"])

# Rolling averages
df["fg3_pct_1"] = df.groupby("player")["fg3_pct"].transform(lambda x: x.rolling(1).mean())
df["fg3_pct_3"] = df.groupby("player")["fg3_pct"].transform(lambda x: x.rolling(3).mean())
df["fg3_pct_5"] = df.groupby("player")["fg3_pct"].transform(lambda x: x.rolling(5).mean())

df["fg3a_1"] = df.groupby("player")["fg3a"].transform(lambda x: x.rolling(1).mean())
df["fg3a_3"] = df.groupby("player")["fg3a"].transform(lambda x: x.rolling(3).mean())
df["fg3a_5"] = df.groupby("player")["fg3a"].transform(lambda x: x.rolling(5).mean())

# Keep only the most recent game per player
latest = df.groupby("player").tail(1)

# Optional: filter to your top-usage players
df_three = (
    data[data["season"] == 2026]
    .sort_values(by="usage", ascending=False)
    .groupby("player")["usage"]
    .max()
    .sort_values(ascending=False)
    .head(110)
    .index
)

latest = latest[latest["player"].isin(df_three)]

# Final view
result = latest[[
    "player","usage",
    "fg3_pct_1", "fg3_pct_3", "fg3_pct_5",
    "fg3a_1", "fg3a_3", "fg3a_5"
]].sort_values(by="usage", ascending=False)

result.head(10)


,player,usage,fg3_pct_1,fg3_pct_3,fg3_pct_5,fg3a_1,fg3a_3,fg3a_5
46141,Tyrese Maxey,0.889931,0.250000,0.250000,0.333333,8.0,6.666667,7.4
45646,Cade Cunningham,0.838889,0.000000,0.222222,0.363333,4.0,4.666667,4.6
46345,Keyonte George,0.822569,0.461538,0.509402,0.372308,13.0,9.666667,7.4
45718,Kevin Durant,0.819792,0.500000,0.458333,0.315000,4.0,6.666667,6.0
45706,Amen Thompson,0.817361,0.000000,0.111111,NaN,2.0,2.000000,1.6
46005,Anthony Edwards,0.816667,0.333333,0.365278,0.422341,15.0,12.000000,10.4
45587,Jalen Johnson,0.812847,0.333333,0.388889,0.261905,6.0,6.000000,5.6
45621,Peyton Watson,0.786458,0.500000,0.333333,0.330000,8.0,4.666667,4.6
46260,Zach LaVine,0.782292,0.250000,0.458333,0.486111,8.0,6.666667,7.0
45602,Donovan Mitchell,0.780903,0.250000,0.310847,0.358730,8.0,8.000000,9.0


# 3p model

In [ ]:
import pandas as pd 

data = 

Processing data/gamelogs/gamelogs_2021.csv
Processing data/gamelogs/gamelogs_2022.csv
Processing data/gamelogs/gamelogs_2023.csv
Processing data/gamelogs/gamelogs_2024.csv
Processing data/gamelogs/gamelogs_2025.csv
Processing data/gamelogs/gamelogs_2026.csv
Total gamelog rows combined: 82627
the lastest date in the data is: 2026-01-21 00:00:00
the total number of unique players is: 802
the total number of gamelog rows is: 46382


In [ ]:
data.to

,player,season,date,team,opp,mp,fg,fga,fg3,fg3a,...,ast,stl,blk,tov,pf,pts,is_home,is_win,mp_minutes,usage
0,Nathan Knight,2021,2020-12-23,ATL,CHI,05:04,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,2.0,2.0,0.0,1,1,5.066667,0.105556
1,Nathan Knight,2021,2020-12-26,ATL,MEM,08:35,4.0,5.0,2.0,3.0,...,0.0,0.0,0.0,1.0,1.0,14.0,1,1,8.583333,0.178819
5,Nathan Knight,2021,2021-01-01,ATL,BRK,00:54,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,0.900000,0.018750
8,Nathan Knight,2021,2021-01-06,ATL,CHO,02:45,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0,0,2.750000,0.057292
9,Nathan Knight,2021,2021-01-09,ATL,CHO,02:07,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1,0,2.116667,0.044097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82605,Corey Kispert,2026,2026-01-19,ATL,MIL,20:02,1.0,4.0,0.0,3.0,...,1.0,1.0,0.0,1.0,1.0,2.0,0,0,20.033333,0.417361
82606,Corey Kispert,2026,2026-01-21,ATL,MEM,13:47,3.0,5.0,2.0,4.0,...,2.0,1.0,0.0,0.0,0.0,8.0,1,1,13.783333,0.287153
82612,Malaki Branham,2026,2026-01-16,WAS,SAC,13:14,6.0,9.0,1.0,2.0,...,0.0,1.0,0.0,0.0,0.0,13.0,1,0,13.233333,0.275694
82616,AJ Johnson,2026,2026-01-16,WAS,SAC,11:28,2.0,6.0,0.0,2.0,...,0.0,1.0,0.0,1.0,1.0,4.0,1,0,11.466667,0.238889


In [17]:
df3[df3['fg3_avg'] > avg_fg3_pct]

,player,fg3_avg,fg3a
0,Nikola Jokić,0.463208,4.754902
1,Kevin Durant,0.453615,5.811881
2,Zach LaVine,0.416697,7.093458
3,Shai Gilgeous-Alexander,0.411929,5.403361
4,Daniss Jenkins(TW),0.408929,2.500000
5,Grayson Allen,0.406952,6.516484
6,Ryan Rollins,0.402349,3.369369
7,Royce O'Neale,0.401545,6.285714
8,A.J. Green,0.400124,5.047297
9,Kawhi Leonard,0.399775,6.089552


In [18]:
df3[df3["fg3a"] > 4].head(15)

,player,fg3_avg,fg3a
0,Nikola Jokić,0.463208,4.754902
1,Kevin Durant,0.453615,5.811881
2,Zach LaVine,0.416697,7.093458
3,Shai Gilgeous-Alexander,0.411929,5.403361
5,Grayson Allen,0.406952,6.516484
7,Royce O'Neale,0.401545,6.285714
8,A.J. Green,0.400124,5.047297
9,Kawhi Leonard,0.399775,6.089552
10,Rui Hachimura,0.399475,4.131868
12,Karl-Anthony Towns,0.398502,4.699115


# pts

In [19]:
df2 = df.groupby('player').agg({'usage': 'max'}).sort_values(by=['usage'], ascending=False, inplace=False).head(110).reset_index()
players = df2["player"].to_list()
df3 = data[data['player'].isin(players)].copy()
df3["fg_avg"] = df3["fg"] / df3["fga"]
fg3a = df3['fga'].mean()
fg3  = df3['fg'].mean()
avg_fg3_pct = fg3 / fg3a
df3 = df3.groupby(['player',"team"]).agg({'fg_avg': 'mean', 'pts': 'mean'}).sort_values(by=['fg_avg'], ascending=False, inplace=False).reset_index()
df3[df3['fg_avg'] > avg_fg3_pct]

,player,team,fg_avg,pts
0,Rudy Gobert,MIN,0.688862,11.640351
1,Jalen Duren,DET,0.662003,11.966667
2,Giannis Antetokounmpo,MIL,0.621273,29.718750
3,Ivica Zubac,LAC,0.611189,16.119658
4,Nikola Jokić,DEN,0.600105,29.598039
5,Domantas Sabonis,SAC,0.590291,18.488095
6,Precious Achiuwa,MIA,0.559895,4.983607
7,Dominick Barlow(TW),PHI,0.558589,8.187500
8,Onyeka Okongwu,ATL,0.552976,11.464286
9,John Collins,LAC,0.552242,13.250000


# ast

In [20]:
df2 = df.groupby('player').agg({'usage': 'max'}).sort_values(by=['usage'], ascending=False, inplace=False).head(110).reset_index()
players = df2["player"].to_list()
df3 = data[data['player'].isin(players)].copy()
#df3["fg_avg"] = df3["fg"] / df3["fga"]
ast_avg = df3['ast'].mean()
df3 = df3.groupby(['player',"team"]).agg({'ast': 'mean', }).sort_values(by=['ast'], ascending=False, inplace=False).reset_index()
df3[df3['ast'] > ast_avg]

,player,team,ast
0,Nikola Jokić,DEN,10.460784
1,Josh Giddey,CHI,9.000000
2,James Harden,LAC,8.470588
3,Luka Dončić,LAL,8.161290
4,Cade Cunningham,DET,7.911765
5,Luka Dončić,DAL,7.818182
6,Russell Westbrook,SAC,6.866667
7,Jalen Brunson,NYK,6.855769
8,Darius Garland,CLE,6.792079
9,Devin Booker,PHO,6.791304
